# Jetson Orin Nano GPIO test

This notebook tests the GPIO-capable pins exposed by `Jetson.GPIO` for a Jetson Orin Nano 40-pin header.

Safety notes before running any output test:

- GPIO pins are 3.3 V logic. Do not connect 5 V to any GPIO pin.
- Use a resistor when driving an LED. A typical test connection is `BOARD pin -> 330 ohm resistor -> LED -> GND`.
- If `Jetson.GPIO` says your carrier board is not verified, confirm the carrier board 40-pin header matches the NVIDIA developer-kit pinout before driving outputs.
- The notebook tests one pin at a time and calls `cleanup()` after each test.

In [1]:
import time
from IPython.display import display

try:
    import pandas as pd
except Exception:
    pd = None

import Jetson.GPIO as GPIO
from Jetson.GPIO import gpio_pin_data

# BOARD numbers on the Jetson Orin Nano 40-pin header that Jetson.GPIO maps as GPIO.
ORIN_NANO_BOARD_PINS = [
    7, 11, 12, 13, 15, 16, 18, 19, 21, 22, 23, 24, 26,
    29, 31, 32, 33, 35, 36, 37, 38, 40,
]

# Static Orin Nano mapping from the installed Jetson.GPIO package.
PIN_META = {
    row[3]: {
        "board": row[3],
        "bcm": row[4],
        "cvm": row[5],
        "tegra_soc": row[6],
        "line_offset": row[0],
        "gpio_name": row[1],
        "gpio_chip": row[2],
        "pwm_chip": row[7],
        "pwm_id": row[8],
        "reg_addr": row[9] if len(row) > 9 else None,
    }
    for row in gpio_pin_data.jetson_gpio_data[gpio_pin_data.JETSON_ORIN_NANO][0]
}

print(f"Jetson.GPIO version: {getattr(GPIO, 'VERSION', 'unknown')}")
print(f"Detected model: {getattr(GPIO, 'model', 'unknown')}")
print(f"Detected info: {getattr(GPIO, 'JETSON_INFO', {})}")

Jetson.GPIO version: 2.1.12
Detected model: JETSON_ORIN_NANO
Detected info: {'P1_REVISION': 1, 'RAM': '32768M, 65536M', 'REVISION': 'Unknown', 'TYPE': 'JETSON_ORIN_NANO', 'MANUFACTURER': 'NVIDIA', 'PROCESSOR': 'A78AE'}


WARNNIG: Jetson.GPIO library has not been verified with this carrier board,


In [ ]:
rows = []
for pin in ORIN_NANO_BOARD_PINS:
    meta = PIN_META[pin]
    rows.append(
        {
            "BOARD": meta["board"],
            "BCM": meta["bcm"],
            "CVM": meta["cvm"],
            "TEGRA_SOC": meta["tegra_soc"],
            "chip": meta["gpio_chip"],
            "line": meta["line_offset"],
            "gpio_name": meta["gpio_name"],
            "pwm": "yes" if meta["pwm_chip"] else "no",
        }
    )

if pd is not None:
    pin_table = pd.DataFrame(rows)
    display(pin_table)
else:
    for row in rows:
        print(row)

In [ ]:
ACTIVE_PINS = set()


def _ensure_board_mode():
    mode = GPIO.getmode()
    if mode is None:
        GPIO.setmode(GPIO.BOARD)
    elif mode != GPIO.BOARD:
        raise RuntimeError("GPIO is already using a non-BOARD numbering mode. Run cleanup() or restart the kernel.")


def _check_pin(pin):
    pin = int(pin)
    if pin not in ORIN_NANO_BOARD_PINS:
        raise ValueError(f"BOARD pin {pin} is not in the Orin Nano GPIO test list: {ORIN_NANO_BOARD_PINS}")
    return pin


def describe_pin(pin):
    pin = _check_pin(pin)
    meta = PIN_META[pin]
    return (
        f"BOARD {pin} | BCM {meta['bcm']} | CVM {meta['cvm']} | "
        f"TEGRA_SOC {meta['tegra_soc']} | {meta['gpio_chip']} line {meta['line_offset']}"
    )


def _setup(pin, direction, initial=None):
    _ensure_board_mode()
    kwargs = {}
    if direction == GPIO.OUT:
        kwargs["initial"] = GPIO.LOW if initial is None else initial
    try:
        GPIO.setup(pin, direction, consumer=f"gpio-test-{pin}", **kwargs)
    except TypeError:
        GPIO.setup(pin, direction, **kwargs)
    ACTIVE_PINS.add(pin)


def cleanup(pin=None):
    pins = list(ACTIVE_PINS) if pin is None else [int(pin)]
    for active_pin in pins:
        try:
            GPIO.output(active_pin, GPIO.LOW)
        except Exception:
            pass
    try:
        if pin is None:
            GPIO.cleanup()
            ACTIVE_PINS.clear()
        else:
            GPIO.cleanup(int(pin))
            ACTIVE_PINS.discard(int(pin))
    except Exception as exc:
        print(f"cleanup warning: {exc}")


def test_output_pin(pin, cycles=5, high_time=0.5, low_time=0.5):
    pin = _check_pin(pin)
    print(f"Testing output: {describe_pin(pin)}")
    _setup(pin, GPIO.OUT, initial=GPIO.LOW)
    try:
        for cycle in range(1, cycles + 1):
            GPIO.output(pin, GPIO.HIGH)
            print(f"  cycle {cycle}: HIGH")
            time.sleep(high_time)
            GPIO.output(pin, GPIO.LOW)
            print(f"  cycle {cycle}: LOW")
            time.sleep(low_time)
    finally:
        cleanup(pin)


def walk_output_test(pins=ORIN_NANO_BOARD_PINS, cycles=3, high_time=0.4, low_time=0.4):
    print("Move your LED, logic probe, or multimeter to each BOARD pin when prompted.")
    print("Press Enter to test, 's' to skip a pin, or 'q' to quit.")
    for pin in pins:
        answer = input(f"{describe_pin(pin)}: ").strip().lower()
        if answer == "q":
            break
        if answer == "s":
            print(f"Skipped BOARD {pin}")
            continue
        test_output_pin(pin, cycles=cycles, high_time=high_time, low_time=low_time)
    cleanup()


def read_input_pin(pin, seconds=10.0, interval=0.05):
    pin = _check_pin(pin)
    print(f"Reading input: {describe_pin(pin)}")
    print("Jetson.GPIO ignores software pull-up/pull-down settings, so use external wiring if needed.")
    _setup(pin, GPIO.IN)
    end_time = time.monotonic() + float(seconds)
    last = None
    try:
        while time.monotonic() < end_time:
            value = GPIO.input(pin)
            if value != last:
                state = "HIGH" if value else "LOW"
                print(f"  {time.strftime('%H:%M:%S')} -> {state}")
                last = value
            time.sleep(interval)
    finally:
        cleanup(pin)


def read_all_inputs_once(pins=ORIN_NANO_BOARD_PINS):
    pins = [_check_pin(pin) for pin in pins]
    _ensure_board_mode()
    GPIO.setup(pins, GPIO.IN)
    ACTIVE_PINS.update(pins)
    try:
        values = [{"BOARD": pin, "state": "HIGH" if GPIO.input(pin) else "LOW"} for pin in pins]
        if pd is not None:
            display(pd.DataFrame(values))
        else:
            for value in values:
                print(value)
        return values
    finally:
        cleanup()


print("Helper functions ready: test_output_pin(), walk_output_test(), read_input_pin(), read_all_inputs_once(), cleanup().")

## Test one output pin

Connect an LED/resistor or meter to the selected physical BOARD pin and a GND pin, then run the cell.

In [ ]:
TEST_PIN = 15
test_output_pin(TEST_PIN, cycles=5, high_time=0.5, low_time=0.5)

## Test USB serial output

Use this instead of GPIO when the output target is connected over USB, such as a USB-to-serial adapter, Arduino, RP2040 board, or another microcontroller. Set `USB_PORT` to the matching `/dev/ttyUSB*` or `/dev/ttyACM*` device.

In [2]:
import time

try:
    import serial
    from serial.tools import list_ports
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError("pyserial is required. Install it with: python -m pip install pyserial") from exc

all_ports = list(list_ports.comports())
usb_ports = [
    port_info
    for port_info in all_ports
    if port_info.device.startswith(("/dev/ttyUSB", "/dev/ttyACM")) or "USB" in port_info.hwid.upper()
]

print("Available USB serial ports:")
if usb_ports:
    for port_info in usb_ports:
        print(f"  {port_info.device}: {port_info.description}")
else:
    print("  none found; plug in the USB serial device and rerun this cell")

# Pick the first detected USB serial port, or set this manually.
USB_PORT = usb_ports[0].device if usb_ports else "/dev/ttyUSB0"
USB_BAUD = 9200
USB_MESSAGES = 10   
USB_INTERVAL = 0.5

# Some USB serial adapters expose DTR/RTS as physical output pins.
# Leave this False for Arduino-style boards unless you intentionally want those lines toggled.
USB_TOGGLE_CONTROL_LINES = False


def test_usb_serial_output(
    port=USB_PORT,
    baudrate=USB_BAUD,
    messages=USB_MESSAGES,
    interval=USB_INTERVAL,
    toggle_control_lines=USB_TOGGLE_CONTROL_LINES,
):
    print(f"Opening {port} at {baudrate} baud")
    with serial.Serial(port, baudrate=baudrate, timeout=1, write_timeout=1) as usb:
        usb.reset_output_buffer()

        if toggle_control_lines:
            for level in (False, True, False):
                usb.setDTR(level)
                usb.setRTS(level)
                print(f"  DTR/RTS -> {'HIGH' if level else 'LOW'}")
                time.sleep(interval)

        for message_number in range(1, messages + 1):
            payload = f"usb-output-test {message_number}\n".encode("utf-8")
            bytes_written = usb.write(payload)
            usb.flush()
            print(f"  wrote {bytes_written} bytes: {payload.decode().strip()}")
            time.sleep(interval)


test_usb_serial_output()

Available USB serial ports:
  none found; plug in the USB serial device and rerun this cell
Opening /dev/ttyUSB0 at 9200 baud


SerialException: [Errno 2] could not open port /dev/ttyUSB0: [Errno 2] No such file or directory: '/dev/ttyUSB0'

## Walk through every GPIO output

This prompts before each pin so you can move the test lead safely.

In [ ]:
walk_output_test(cycles=3, high_time=0.4, low_time=0.4)

## Test one input pin

Use a safe 3.3 V signal or a jumper through appropriate protection. Do not feed 5 V into the pin.

In [ ]:
INPUT_PIN = 15
read_input_pin(INPUT_PIN, seconds=10)

## Read all GPIO inputs once

This configures the GPIO-capable header pins as inputs, reads each once, and cleans up.

In [ ]:
read_all_inputs_once()

## Cleanup

Run this if you interrupt a test cell.

In [ ]:
cleanup()

In [1]:
import time
import serial

PORT = "/dev/ttyUSB0"   # Linux/Raspberry Pi
# PORT = "COM3"         # Windows example

ser = serial.Serial(PORT, 9600, timeout=1)

relay_on  = bytes([0xA0, 0x01, 0x01, 0xA2])
relay_off = bytes([0xA0, 0x01, 0x00, 0xA1])

print("Relay ON")
ser.write(relay_on)
time.sleep(5)

print("Relay OFF")
ser.write(relay_off)
time.sleep(1)

ser.close()
print("Done")

SerialException: [Errno 13] could not open port /dev/ttyUSB0: [Errno 13] Permission denied: '/dev/ttyUSB0'